<div align="center">

<!-- MOTIONSALT branded banner. Rendered as HTML for the logo mark + gradient. -->
<div style="background:linear-gradient(135deg,#0f172a 0%,#1e1b4b 60%,#312e81 100%);padding:28px 24px;border-radius:14px;color:#f8fafc;font-family:-apple-system,Segoe UI,Roboto,sans-serif;">
  <div style="display:flex;align-items:center;justify-content:center;gap:14px;">
    <div style="width:44px;height:44px;border-radius:10px;background:linear-gradient(135deg,#22d3ee,#a855f7);display:flex;align-items:center;justify-content:center;font-weight:900;font-size:22px;color:#0f172a;">M</div>
    <div style="font-size:30px;font-weight:800;letter-spacing:2px;">MOTIONSALT</div>
  </div>
  <div style="margin-top:8px;font-size:14px;opacity:0.85;letter-spacing:3px;text-transform:uppercase;">Anime&nbsp;Video&nbsp;Upscaler</div>
  <div style="margin-top:14px;font-size:14px;opacity:0.75;max-width:640px;margin-left:auto;margin-right:auto;">A free, no-install, GPU-in-the-cloud alternative to Topaz Video AI. Powered by AnimeJaNai&nbsp;V3 and Real-ESRGAN AnimeVideo&nbsp;v3.</div>
  <div style="margin-top:18px;font-size:12px;opacity:0.7;">
    <a style="color:#a5f3fc;text-decoration:none;" href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
  </div>
</div>

</div>

---

**How this works:** step through the four cells below in order. Each cell is a self-contained step of a wizard — Connect ➜ Upload ➜ Configure ➜ Download. You never need to read or edit any code.

## Step 1 — Connect

Click **Connect** below. This verifies your GPU, installs the dependencies, and downloads the AI model weights from the MOTIONSALT GitHub Releases (never from HuggingFace — see the README for why).

In [ ]:
#@title 🔌 Step 1 — Connect { display-mode: "form" }
#@markdown Click the **Connect** button that appears below this cell after you run it.
import os, sys, subprocess, shutil, json, time, urllib.request, urllib.error
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------- MOTIONSALT global state ----------
MS = globals().setdefault("MOTIONSALT", {})
MS.setdefault("workdir", Path("/content/motionsalt"))
MS["workdir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("weights_dir", MS["workdir"] / "weights")
MS["weights_dir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("connected", False)

# ---------- Config: where to pull weights from ----------
# The Colab notebook ONLY ever downloads weights from this GitHub repo's Releases.
# The HuggingFace upstream is mirrored by a scheduled Action; the notebook itself
# never contacts HuggingFace directly.
GH_REPO   = "motionssalt/upscale"          # <-- change to your fork
GH_TAG    = "latest"                                    # "latest" resolves to the newest weights-vX.Y.Z tag
WEIGHTS = {
    "LOW":    "2x_AnimeJaNaiV3_SuperUltraCompact.pth",
    "MEDIUM": "2x_AnimeJaNaiV3_UltraCompact.pth",
    "HIGH":   "realesr-animevideov3.pth",
}
MS["weights_map"] = WEIGHTS
MS["gh_repo"]     = GH_REPO

# ---------- Branded status log ----------
_log = widgets.HTML(value="")
_lines = []
def status(kind, msg):
    icon = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•"}[kind]
    color = {"ok":"#16a34a","warn":"#d97706","err":"#dc2626","run":"#0891b2","info":"#475569"}[kind]
    _lines.append(f'<div style="font-family:ui-monospace,Menlo,monospace;font-size:12.5px;color:{color};padding:2px 0;">{icon}&nbsp;&nbsp;{msg}</div>')
    _log.value = "".join(_lines)

def section(title):
    _lines.append(f'<div style="margin:10px 0 4px 0;font-weight:600;color:#0f172a;font-family:-apple-system,Segoe UI,sans-serif;">{title}</div>')
    _log.value = "".join(_lines)

# ---------- The Connect button ----------
btn = widgets.Button(
    description="Connect",
    icon="plug",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="42px"),
)
badge = widgets.HTML(
    '<span style="display:inline-block;padding:4px 10px;border-radius:999px;background:#e2e8f0;color:#334155;font-size:11px;font-family:-apple-system,sans-serif;">not connected</span>'
)
header = widgets.HTML(
    '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Connect to a MOTIONSALT session</div>'
    '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:8px;">Verifies GPU · installs dependencies · pulls model weights from GitHub Releases</div>'
)

def _run(cmd, quiet=True):
    """Run a shell command; return (returncode, tail_of_output)."""
    p = subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True, text=True)
    if not quiet and p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-2000:])
    return p.returncode, (p.stderr or p.stdout)[-400:]

def _resolve_release_tag():
    """Ask GitHub for the newest release tag (unauthenticated is fine — public repo)."""
    if GH_TAG != "latest":
        return GH_TAG
    url = f"https://api.github.com/repos/{GH_REPO}/releases/latest"
    with urllib.request.urlopen(url, timeout=20) as r:
        data = json.loads(r.read().decode("utf-8"))
    return data["tag_name"]

def _download(url, dest: Path, label: str):
    """Streaming download with a live progress bar."""
    bar = widgets.IntProgress(value=0, min=0, max=100, description=label,
                              layout=widgets.Layout(width="100%"),
                              bar_style="info")
    pct = widgets.HTML(value="0%")
    row = widgets.HBox([bar, pct])
    display(row)
    req = urllib.request.Request(url, headers={"User-Agent":"motionsalt-upscaler"})
    with urllib.request.urlopen(req, timeout=60) as r:
        total = int(r.headers.get("Content-Length", "0")) or 0
        read = 0
        with dest.open("wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk); read += len(chunk)
                if total:
                    p = int(read * 100 / total)
                    bar.value = p; pct.value = f"{p}%"
        if total:
            bar.value = 100; pct.value = "100%"
        bar.bar_style = "success"

def on_connect(_):
    btn.disabled = True
    badge.value = '<span style="display:inline-block;padding:4px 10px;border-radius:999px;background:#fef3c7;color:#92400e;font-size:11px;">connecting…</span>'
    _lines.clear(); _log.value = ""

    # 1. GPU
    section("1 / 4 · GPU")
    try:
        import torch
        if not torch.cuda.is_available():
            status("err", "No CUDA GPU detected. Enable GPU: Runtime → Change runtime type → GPU.")
            badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#fee2e2;color:#991b1b;font-size:11px;">no GPU</span>'
            btn.disabled = False
            return
        name = torch.cuda.get_device_name(0)
        status("ok", f"GPU detected: <b>{name}</b>")
    except Exception as e:
        status("err", f"PyTorch not importable yet: {e}")

    # 2. System deps (ffmpeg)
    section("2 / 4 · System dependencies")
    if shutil.which("ffmpeg"):
        status("ok", "ffmpeg already present.")
    else:
        status("run", "Installing ffmpeg…")
        rc, tail = _run("apt-get -qq update && apt-get -qq install -y ffmpeg")
        status("ok" if rc==0 else "err", "ffmpeg installed." if rc==0 else f"ffmpeg install failed: {tail}")

    # 3. Python deps
    section("3 / 4 · Python packages")
    pkgs = ["opencv-python-headless", "numpy", "spandrel", "tqdm"]
    # spandrel: universal loader that supports both AnimeJaNai (compact/SRVGGNet) and Real-ESRGAN (SRVGGNet) checkpoints.
    status("run", "Installing " + ", ".join(pkgs) + " …")
    rc, tail = _run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    status("ok" if rc==0 else "err", "Python packages ready." if rc==0 else f"pip failed: {tail}")

    # 4. Weights — from THIS GitHub repo, never HuggingFace
    section("4 / 4 · Model weights (from GitHub Releases)")
    try:
        tag = _resolve_release_tag()
        status("ok", f"Resolved release tag: <b>{tag}</b>")
    except Exception as e:
        status("err", f"Could not reach GitHub API: {e}")
        btn.disabled = False; return

    ok_all = True
    for tier, fname in WEIGHTS.items():
        dest = MS["weights_dir"] / fname
        if dest.exists() and dest.stat().st_size > 0:
            status("ok", f"{tier} — {fname} already cached.")
            continue
        url = f"https://github.com/{GH_REPO}/releases/download/{tag}/{fname}"
        status("run", f"{tier} — downloading {fname}…")
        try:
            _download(url, dest, tier)
            status("ok", f"{tier} — downloaded.")
        except Exception as e:
            status("err", f"{tier} — download failed: {e}")
            ok_all = False

    if ok_all:
        MS["connected"] = True
        MS["release_tag"] = tag
        badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#dcfce7;color:#166534;font-size:11px;">connected</span>'
        status("ok", "<b>Ready.</b> Continue to Step 2.")
    else:
        badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#fee2e2;color:#991b1b;font-size:11px;">error</span>'

    btn.disabled = False

btn.on_click(on_connect)

panel = widgets.VBox([
    header,
    widgets.HBox([btn, badge]),
    _log,
], layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px"))
display(panel)


## Step 2 — Upload your video

Pick a video file from your device. It's copied into the Colab VM only — nothing is sent to a third-party service.

In [ ]:
#@title 📤 Step 2 — Upload video { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path
import shutil, os

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("connected"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 1 first (Connect).</div>'))
else:
    header = widgets.HTML(
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Upload the video you want to upscale</div>'
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;">MP4, MKV, MOV, WebM, AVI… anything ffmpeg reads.</div>'
    )
    pick_btn = widgets.Button(description="Choose file…", icon="upload",
                              button_style="primary",
                              layout=widgets.Layout(width="180px", height="42px"))
    prog = widgets.IntProgress(value=0, min=0, max=100, description="Copy",
                               layout=widgets.Layout(width="100%"),
                               bar_style="info")
    prog_pct = widgets.HTML("0%")
    prog_row = widgets.HBox([prog, prog_pct])
    prog_row.layout.display = "none"
    result = widgets.HTML("")

    def on_pick(_):
        # Use the Colab file uploader; then copy into workspace with progress.
        from google.colab import files
        pick_btn.disabled = True
        result.value = '<div style="font-family:sans-serif;font-size:12.5px;color:#475569;">Waiting for browser file picker…</div>'
        uploaded = files.upload()
        if not uploaded:
            result.value = '<div style="padding:8px;background:#fef3c7;color:#92400e;border-radius:6px;font-family:sans-serif;">No file selected.</div>'
            pick_btn.disabled = False
            return
        name, data = next(iter(uploaded.items()))
        src_tmp = Path("/content") / name
        # google.colab.files.upload() already wrote it into /content; we just move it.
        dest = MS["workdir"] / "input" / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        prog_row.layout.display = "flex"
        # Copy with progress (chunked, so the bar is real, not fake).
        total = len(data); read = 0
        with open(src_tmp, "rb") as fin, open(dest, "wb") as fout:
            while True:
                chunk = fin.read(1 << 20)
                if not chunk: break
                fout.write(chunk); read += len(chunk)
                prog.value = int(read * 100 / max(total, 1))
                prog_pct.value = f"{prog.value}%"
        prog.value = 100; prog_pct.value = "100%"; prog.bar_style = "success"
        try: os.remove(src_tmp)
        except OSError: pass
        MS["input_path"] = dest
        size_mb = dest.stat().st_size / 1e6
        result.value = (
            f'<div style="padding:10px;background:#dcfce7;color:#166534;border-radius:8px;font-family:sans-serif;">'
            f'✅ Uploaded <b>{name}</b> ({size_mb:.1f} MB). Continue to Step 3.'
            f'</div>'
        )
        pick_btn.disabled = False

    pick_btn.on_click(on_pick)
    display(widgets.VBox([header, pick_btn, prog_row, result],
                         layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


## Step 3 — Configure & process

Pick a quality tier and dial in the filters. Then click **Start Processing**. A progress bar shows real frame-by-frame progress.

In [ ]:
#@title ⚙️ Step 3 — Configure & process { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path
import subprocess, shutil, math, json, time, os, sys, threading, collections, queue

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("connected"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 1 first (Connect).</div>'))
elif not MS.get("input_path"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 2 first (Upload).</div>'))
else:
    # -------- Widgets --------
    # NOTE (Bug 0): the model dropdown intentionally does NOT print a fixed
    # "N×" in its labels. Native scale is read off the loaded model at runtime
    # (spandrel's ImageModelDescriptor exposes .scale), so nothing hardcodes 2×.
    tier = widgets.Dropdown(
        options=[
            ("LOW  — AnimeJaNai V3 SuperUltraCompact (fastest)", "LOW"),
            ("MEDIUM — AnimeJaNai V3 UltraCompact (balanced)", "MEDIUM"),
            ("HIGH — Real-ESRGAN AnimeVideo v3 (best quality)", "HIGH"),
        ],
        value="MEDIUM",
        description="Quality",
        style={"description_width": "160px"},
        layout=widgets.Layout(width="640px"),
    )

    def slider(desc, default=0):
        return widgets.IntSlider(
            value=default, min=0, max=100, step=1, description=desc,
            style={"description_width": "160px"},
            layout=widgets.Layout(width="640px"),
            continuous_update=False,
        )

    s_revert    = slider("Revert Compression", 0)
    s_detail    = slider("Improve Detail", 0)
    s_sharpen   = slider("Sharpen", 15)
    s_denoise   = slider("Reduce Noise", 0)
    s_dehalo    = slider("Dehalo", 0)
    s_deblur    = slider("Anti-alias/Deblur", 0)
    s_recover   = slider("Recover Original Detail", 0)
    cb_1080     = widgets.Checkbox(value=False, description="Downscale to 1080p height (preserve aspect ratio)",
                                   indent=False)
    # BUGFIX (fp16, kept removed): the fp16 toggle stays gone. Previous debug
    # rounds kept flipping fp16 on/off as a "just in case" lever and it made
    # zero measurable difference to end-to-end fps — the bottleneck was never
    # the model's math precision. The profiling pass confirmed this: the model
    # forward pass (infer_kernel) is only ~8-13% of frame time while ~85% is
    # CPU-side pre/post/recover work. Everything below runs in fp32. If a
    # future pass wants to reintroduce half precision, do it behind a Boolean
    # that is explicitly validated against the profiler output, not a UI toggle.

    start = widgets.Button(description="Start Processing", icon="play",
                           button_style="success",
                           layout=widgets.Layout(width="220px", height="44px"))
    frame_prog = widgets.IntProgress(value=0, min=0, max=100, description="Frames",
                                     layout=widgets.Layout(width="100%"), bar_style="info")
    frame_pct  = widgets.HTML("0%")
    stage_lbl  = widgets.HTML("")
    log        = widgets.HTML("")
    _msgs = []
    def logline(kind, msg):
        icon = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•","perf":"📊"}[kind]
        color = {"ok":"#16a34a","warn":"#d97706","err":"#dc2626","run":"#0891b2","info":"#475569","perf":"#7c3aed"}[kind]
        _msgs.append(f'<div style="font-family:ui-monospace,Menlo,monospace;font-size:12.5px;color:{color};padding:2px 0;">{icon}&nbsp;&nbsp;{msg}</div>')
        log.value = "".join(_msgs)
        # Also mirror perf/info lines to stdout so they survive kernel restarts
        # and can be grepped out of the notebook JSON afterwards.
        if kind in ("perf", "warn", "err", "ok"):
            print(f"[motionsalt/{kind}] {msg}", flush=True)

    hdr = widgets.HTML(
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Configure processing</div>'
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;">Defaults are safe. All sliders are 0–100.</div>'
    )

    def start_processing(_):
        start.disabled = True
        _msgs.clear(); log.value = ""
        try:
            _do_process()
        except Exception as e:
            logline("err", f"Processing failed: {e}")
            raise
        finally:
            start.disabled = False

    # ---------- The actual pipeline ----------
    def _ffprobe(path, args):
        out = subprocess.check_output(["ffprobe","-v","error", *args, str(path)]).decode().strip()
        return out

    def _nvidia_smi_snapshot():
        """One-shot GPU util + mem snapshot via nvidia-smi. Returns a short
        string like 'gpu=87% mem=3421/15109MiB' or '' if nvidia-smi is missing.
        Cheap enough to call every few seconds; we do NOT call it in the hot
        loop. Used to prove-or-disprove the 'GPU is idle' hypothesis without
        making assumptions."""
        try:
            out = subprocess.check_output(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used,memory.total",
                 "--format=csv,noheader,nounits"],
                stderr=subprocess.DEVNULL, timeout=1.5,
            ).decode().strip().splitlines()[0]
            util, used, total = [x.strip() for x in out.split(",")]
            return f"gpu={util}% mem={used}/{total}MiB"
        except Exception:
            return ""

    def _load_model(tier_val):
        """Load the checkpoint for the chosen tier via spandrel. Always fp32
        (the fp16 lever was removed — see note above).

        Returns (descriptor, scale). We read `.scale` off the loaded
        ImageModelDescriptor so nothing downstream has to hardcode a factor.
        """
        import torch
        from spandrel import ModelLoader, ImageModelDescriptor
        weight_file = MS["weights_map"][tier_val]
        path = MS["weights_dir"] / weight_file
        model = ModelLoader().load_from_file(str(path))
        if not isinstance(model, ImageModelDescriptor):
            raise RuntimeError(
                f"{weight_file} did not load as an ImageModelDescriptor "
                f"(got {type(model).__name__}) — cannot use this checkpoint.")
        model.cuda().eval()
        scale = int(getattr(model, "scale", 0)) or 0
        if scale <= 0:
            raise RuntimeError(
                f"Loaded {weight_file} but could not determine its upscale "
                f"factor from the descriptor (.scale={scale!r}). Refusing to "
                f"guess — that is exactly the assumption that broke before.")
        return model, scale

    # ------------------------------------------------------------------
    # GPU-RESIDENT PIPELINE — bug-fix pass on top of the earlier port
    # ------------------------------------------------------------------
    # The earlier pass moved pre/post/recover onto GPU tensors and killed the
    # CPU-bound idle time (pre 6172ms -> tens of ms, GPU util 0% -> loaded).
    # That fix is kept in full — we do NOT revert to CPU filters.
    #
    # This pass fixes two follow-on errors that showed up after the port:
    #
    # ERROR 1: "Inplace update to inference tensor outside InferenceMode is
    #           not allowed. You can make a clone to get a normal tensor
    #           before doing inplace update."
    #
    #   Cause: the model forward ran inside a narrow `torch.no_grad()`
    #   (inside _infer_kernel_only), but the returned tensor `y` was then
    #   handed to `_recover_blend_gpu`, which called `up.lerp_(...)` — an
    #   in-place op — OUTSIDE the no_grad/inference context. On PyTorch
    #   builds where the model's forward runs under inference-mode semantics
    #   (spandrel wraps some models this way), the output is an "inference
    #   tensor" and mutating it outside inference mode is a hard error.
    #
    #   Fix: run the ENTIRE per-frame GPU pipeline (H2D copy + pre + infer +
    #   recover + post + D2H) under a single `torch.inference_mode()`
    #   context. That is the recommended, permanent fix — not clone-on-write
    #   guessing per call site. Every intermediate is an inference tensor
    #   consistently, so all in-place ops (lerp_, div_, etc.) are legal.
    #   We also drop the redundant `torch.no_grad()` inside the infer
    #   helper — inference_mode is strictly stronger and enabling both is
    #   just noise.
    #
    # ERROR 2: "CUDA out of memory. Tried to allocate 1.14 GiB. 3.43 GiB is
    #           reserved by PyTorch but unallocated."  (T4, 14.56 GiB total.)
    #
    #   Cause: three memory bombs in the ported filters, all triggered on
    #   HIGH tier where the input reaches 1080p / 4K-class frames:
    #
    #     (a) bilateral in _pre_filters_gpu did
    #             xp.unfold(2, d, 1).unfold(3, d, 1)  # 1,3,H,W,d,d
    #         At d=9 for a 1080p input that single tensor is
    #             1*3*1080*1920*9*9*4 = 6.05 GiB.
    #         Plus the same-shape `diff` and `w` tensors it multiplies with.
    #
    #     (b) NLM-lite did the same trick with a 7x7 search window and then
    #         reshape/permute to (49, 3, H, W) — another multi-GiB spike.
    #
    #     (c) CLAHE built four full-resolution (1,1,H,W) lookup gathers on
    #         the UPSCALED frame (H*scale, W*scale), which on HIGH (4x) is
    #         another few hundred MB.
    #
    #   The 3.43 GiB "reserved but unallocated" is the classic PyTorch
    #   caching-allocator fragmentation signature after these giant one-shot
    #   tensors are freed and the allocator can't find a contiguous block.
    #
    #   Fix (all permanent, not per-frame-guarded):
    #     * Both unfold-based filters are rewritten as a **shift-and-accumulate
    #       loop** over the (dy, dx) offsets of the window. This is
    #       numerically identical to the unfold version — same weights, same
    #       math — but peak memory is O(H*W*channels) instead of
    #       O(H*W*channels*d*d). One frame at HIGH tier now peaks well under
    #       500 MB in the filter stages instead of 6+ GiB.
    #     * CLAHE runs its bilinear-gather math in-place with a small helper
    #       that gathers ONE tile-LUT at a time and accumulates, avoiding
    #       four full-frame intermediates.
    #     * We enable the expandable-segments allocator (env var
    #       PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True) at process
    #       start so the caching allocator does not fragment across the
    #       larger-than-usual filter allocations. It's a no-op on PyTorch
    #       builds that don't support it.
    #     * After model load and after the very first frame (warm-up),
    #       torch.cuda.empty_cache() is called ONCE to release the transient
    #       cudnn.benchmark scratch. Not per-frame.
    #     * cudnn.benchmark is left on (it earned its keep in the profile),
    #       but we no longer let intermediate filter tensors leak references
    #       — each stage explicitly `del`s its input alias before returning.
    # ------------------------------------------------------------------

    def _bgr_u8_to_rgb_f32(bgr_u8_gpu):
        """(H,W,3) uint8 BGR CUDA tensor -> (1,3,H,W) fp32 RGB CUDA tensor.

        The channel reversal (BGR->RGB), HWC->CHW permute, batch unsqueeze,
        and /255 normalization all happen on-GPU. flip() + permute() are
        views until the final .contiguous() makes one coalesced GPU copy —
        replacing what used to be a CPU-side np.ascontiguousarray copy."""
        import torch
        return (bgr_u8_gpu
                .flip(-1)                 # BGR -> RGB (view)
                .permute(2, 0, 1)         # HWC -> CHW (view)
                .unsqueeze(0)             # -> NCHW
                .contiguous()             # one GPU kernel, coalesced
                .to(torch.float32)
                .div_(255.0))

    def _rgb_f32_to_bgr_u8(x):
        """(1,3,H,W) fp32 RGB CUDA tensor -> (H,W,3) uint8 BGR CUDA tensor.

        clamp/round happen on-GPU so the single D2H copy at the end moves the
        smallest possible payload (uint8, 3 bytes/px) instead of fp32.

        Non-mutating on the input on purpose: the input may still be aliased
        elsewhere in the pipeline (recover holds `t_in` for the blend)."""
        import torch
        return (x.clamp(0.0, 1.0)
                 .mul(255.0)
                 .round()
                 .to(torch.uint8)
                 .squeeze(0)
                 .permute(1, 2, 0)         # CHW -> HWC (view)
                 .flip(-1)                 # RGB -> BGR (view)
                 .contiguous())            # one GPU kernel, packed for D2H

    def _gaussian_blur_gpu(x, sigma, radius=None):
        """OpenCV-compatible separable Gaussian blur on an (1,3,H,W) fp32
        CUDA tensor. Matches cv2.GaussianBlur(x, (0,0), sigma): kernel size is
        derived from sigma the same way OpenCV derives it, and the border is
        REFLECT_101 (cv2.BORDER_DEFAULT) via F.pad(mode='reflect').

        Separable: two 1D conv2ds instead of one 2D. Peak intermediate is a
        single (1,3,H,W) tensor, not (1,3,H,W,k,k) — matters at 4K on HIGH."""
        import torch, torch.nn.functional as F
        if sigma <= 0:
            return x
        # cv2.getGaussianKernel: ksize = round(sigma*4*2 + 1) | 1 when ksize=0
        if radius is None:
            ksize = int(round(sigma * 4.0 * 2.0 + 1.0)) | 1
            radius = (ksize - 1) // 2
        r = torch.arange(-radius, radius + 1, device=x.device, dtype=x.dtype)
        k = torch.exp(r * r / (-2.0 * sigma * sigma))
        k = k / k.sum()
        kh = k.view(1, 1, 1, -1).expand(3, 1, 1, -1)
        kv = k.view(1, 1, -1, 1).expand(3, 1, -1, 1)
        xp = F.pad(x, (radius, radius, 0, 0), mode="reflect")
        x_h = F.conv2d(xp, kh, groups=3)
        del xp
        xp = F.pad(x_h, (0, 0, radius, radius), mode="reflect")
        del x_h
        return F.conv2d(xp, kv, groups=3)

    def _pre_filters_gpu(t, params, _cache={}):
        """Pre-inference cleanup, entirely on-GPU. Input/output are
        (1,3,H,W) fp32 RGB CUDA tensors in [0,1].

        revert  — bilateral deblock. Previously cv2.bilateralFilter on CPU.
                  Now: shift-and-accumulate over the d×d window offsets.
                  Numerically identical to the unfold version (same window,
                  same Gaussian range & space weights) but peak memory is
                  O(H*W*3) per shift instead of O(H*W*3*d*d) — this is the
                  Error-2 OOM fix at HIGH tier.

        denoise — NLM-lite: 3x3-patch distance over a 7x7 search window,
                  computed as a shift loop over the 49 (dy,dx) offsets. The
                  patch distance is what preserves line-art under strong
                  denoise; we keep it, but never materialize the full
                  (1,3,H,W,7,7) tensor. Same knob as before."""
        import torch, torch.nn.functional as F
        out = t
        if params["revert"] > 0:
            k = params["revert"] / 100.0
            d = int(3 + 6 * k)                          # same knob as before
            # cv2.bilateralFilter with diameter d uses a d×d window anchored
            # at d//2 — for even d that is offsets -(d//2)..(d//2 - 1), NOT a
            # symmetric window. Match that exactly.
            lo = d // 2                                 # offsets -lo .. d-lo-1
            hi = d - lo - 1
            sigma_color = (20.0 + 60.0 * k) / 255.0     # fp32 [0,1] domain
            sigma_space = 20.0 + 40.0 * k
            two_sc2 = 2.0 * sigma_color * sigma_color
            two_ss2 = 2.0 * sigma_space * sigma_space

            # Reflect-pad ONCE, then slice per (dy, dx) — cheap views into the
            # same padded buffer. Peak allocation per shift is one (1,3,H,W)
            # weight tensor and one running numerator/denominator.
            xp = F.pad(out, (lo, hi, lo, hi), mode="reflect")
            _, _, Hp, Wp = xp.shape
            _, _, H, W = out.shape
            num = torch.zeros_like(out)
            den = torch.zeros((1, 1, H, W), device=out.device, dtype=out.dtype)
            for dy in range(-lo, hi + 1):
                for dx in range(-lo, hi + 1):
                    # shifted neighborhood pixel: xp[:, :, lo+dy : lo+dy+H, lo+dx : lo+dx+W]
                    nb = xp[:, :, lo + dy: lo + dy + H, lo + dx: lo + dx + W]
                    # color weight (per-pixel scalar)
                    diff = nb - out
                    dist_c = (diff * diff).sum(dim=1, keepdim=True)
                    # space weight (a single float — cached wouldn't help)
                    gs = math.exp(-(dx * dx + dy * dy) / two_ss2)
                    w = torch.exp(dist_c / (-two_sc2)) * gs
                    num.add_(nb * w)
                    den.add_(w)
                    del diff, dist_c, w
            out = num / den.clamp_min(1e-8)
            del xp, num, den

        if params["denoise"] > 0:
            # NLM-lite: non-local weighted mean over a 7x7 search window,
            # 3x3-patch distances. Shift-and-accumulate to keep peak memory
            # at O(H*W*3) — the old unfold version peaked at
            # ~3.6 GiB on a 1080p frame from the (1,3,H,W,7,7) tensor.
            #
            # Numerical parity note: the reference implementation ran
            # `_box_blur_3x3` on each shifted (1,3,H,W) slice INDEPENDENTLY
            # (each shift got its own replicate-padded borders). Blurring
            # the padded parent once and slicing is NOT equivalent — the
            # border pixels of each shifted slice differ because they come
            # from the parent's real content rather than replicate padding.
            # So we do the per-slice blur here; it's still O(H*W*3) peak,
            # bounded, but numerically identical to the reference.
            h_par = 3.0 + 12.0 * (params["denoise"] / 100.0)
            sigma_c = max(h_par, 1.0) * 1.5 / 255.0
            two_sc2 = 2.0 * sigma_c * sigma_c
            radius = 3                                  # 7x7 search window
            two_ss2 = 2.0 * 2.0 * 2.0                   # gaussian sigma 2 (space)
            center_b = _box_blur_3x3(out)               # (1,3,H,W)

            _, _, H, W = out.shape
            xp = F.pad(out, (radius,) * 4, mode="reflect")
            num = torch.zeros_like(out)
            den = torch.zeros((1, 1, H, W), device=out.device, dtype=out.dtype)
            for dy in range(-radius, radius + 1):
                for dx in range(-radius, radius + 1):
                    nb = xp[:, :, radius + dy: radius + dy + H,
                                  radius + dx: radius + dx + W]
                    # Per-slice box blur — matches the reference's
                    # replicate-padded 3x3 mean applied to each shift
                    # independently. `nb` is a slice view; box_blur_3x3
                    # takes a .contiguous copy via F.pad internally.
                    nb_b = _box_blur_3x3(nb)
                    diff = nb_b - center_b
                    dist = (diff * diff).sum(dim=1, keepdim=True)
                    del diff, nb_b
                    gs = math.exp(-(dx * dx + dy * dy) / two_ss2)
                    w = torch.exp(dist / (-two_sc2)) * gs
                    del dist
                    num.add_(nb * w)
                    den.add_(w)
                    del w
            out = num / den.clamp_min(1e-8)
            del xp, num, den, center_b
        return out

    def _recover_blend_gpu(up, src, strength_0_100, scale, _cache={}):
        """Recover Original Detail, entirely on-GPU.

        Was: cv2.resize(original, (w*scale, h*scale), INTER_LANCZOS4) on CPU
        (a full 4K lanczos resize per frame — the 1.2 s `recover` bucket)
        followed by a numpy fp32 blend. Now: F.interpolate on the GPU tensor
        + an in-place torch lerp.

        BUGFIX (Error 1): this used to call `up.lerp_(...)` unconditionally,
        which errored when `up` was an inference tensor produced by the
        model forward and this function was called OUTSIDE inference_mode.
        The whole per-frame pipeline now runs inside a single
        torch.inference_mode() block (see _do_process), so `up` and `naive`
        are BOTH inference tensors and the in-place lerp is legal. As a
        belt-and-braces guard we still fall back to an out-of-place lerp if
        for any reason `up` is not writable in place."""
        if strength_0_100 <= 0:
            return up
        import torch, torch.nn.functional as F
        _, _, H, W = up.shape
        naive = F.interpolate(src, size=(H, W), mode="bicubic",
                              align_corners=False, antialias=True)
        alpha = 0.5 * (strength_0_100 / 100.0)          # same knob as before
        try:
            return up.lerp_(naive, alpha)
        except RuntimeError:
            # Extremely defensive: only trips on exotic PyTorch builds that
            # still refuse in-place under inference_mode. Numerically
            # identical to the in-place path.
            return up.lerp(naive, alpha)

    def _box_blur_3x3(x):
        """3x3 mean filter via summed-area table (integral image). O(1) per
        pixel regardless of what wraps it; borders use replicate padding,
        matching how the old CPU path treated the 1-px edge."""
        import torch, torch.nn.functional as F
        xp = F.pad(x, (1, 1, 1, 1), mode="replicate")
        integ = xp.cumsum(2).cumsum(3)
        del xp
        integ = F.pad(integ, (1, 0, 1, 0))              # zero row/col on top/left
        s = (integ[:, :, 3:, 3:] - integ[:, :, :-3, 3:]
             - integ[:, :, 3:, :-3] + integ[:, :, :-3, :-3])
        return s / 9.0

    def _clahe_l_gpu(L, clip_limit, tiles=(8, 8)):
        """Contrast-limited adaptive histogram equalization on the L channel,
        on-GPU. L is (1,1,H,W) fp32 in [0,1]. Mirrors cv2.createCLAHE:
        per-tile histograms, clip with redistribution, per-tile CDF LUTs,
        bilinear interpolation of neighboring tile LUTs per pixel.

        OpenCV's exact tile-edge sampler differs at the sub-pixel level, so
        this is visually identical to the old CPU path rather than bit-exact;
        it only runs when Improve Detail > 0.

        MEMORY FIX (Error 2): the previous implementation gathered FOUR full
        (1,1,H,W) LUT lookups (l00/l01/l10/l11) before blending — four
        upscaled-resolution intermediates that pushed the T4 over the edge
        on HIGH tier. Now we accumulate the bilinear blend into a single
        output tensor, one corner at a time, freeing each gather as we go."""
        import torch, torch.nn.functional as F
        _, _, H, W = L.shape
        tx, ty = tiles
        dev, dt = L.device, L.dtype

        # Quantize to 256 bins. round() matches cv2's float->uint8 L channel
        # conversion (round-to-nearest), which plain truncation does not.
        q = (L.clamp(0, 1) * 255.0).round().long()

        # --- per-tile clipped CDF LUTs (8x8 tiles -> tiny loops, all GPU) ---
        tw = (W + tx - 1) // tx
        th = (H + ty - 1) // ty
        lut = torch.empty(ty, tx, 256, device=dev, dtype=dt)
        hist_bins = torch.arange(256, device=dev)
        for gy in range(ty):
            for gx in range(tx):
                tile = q[:, :, gy * th:(gy + 1) * th, gx * tw:(gx + 1) * tw]
                n = tile.numel()
                if n == 0:
                    lut[gy, gx] = hist_bins.to(dt) / 255.0
                    continue
                hist = torch.bincount(tile.reshape(-1), minlength=256).to(dt)
                # clip + redistribute (cv2: clipLimit vs average-per-bin)
                limit = max(clip_limit * n / 256.0, 1.0)
                excess = (hist - limit).clamp_min(0).sum()
                hist = hist.clamp(max=limit)
                hist = hist + excess / 256.0            # uniform redistribution
                cdf = hist.cumsum(0)
                # cv2's LUT is cvRound(cdf[i] * 255 / n) — verified against
                # OpenCV behavior (a 2-bin step tile maps to 4 and 255, which
                # only this formula reproduces).
                lut[gy, gx] = (cdf * (255.0 / max(n, 1))).round() / 255.0

        # --- per-pixel bilinear blend of the 4 nearest tile LUTs ---
        ys = torch.arange(H, device=dev, dtype=dt)
        xs = torch.arange(W, device=dev, dtype=dt)
        # cv2's sampler maps pixel y -> y/th - 0.5 (NOT (y+0.5)/th - 0.5):
        # each tile's LUT is centered on that tile's middle pixel row/col.
        gyf = ys / float(th) - 0.5
        gxf = xs / float(tw) - 0.5
        gy0f = gyf.floor()
        gx0f = gxf.floor()
        fy = (gyf - gy0f).view(1, 1, H, 1)
        fx = (gxf - gx0f).view(1, 1, 1, W)
        gy0 = gy0f.clamp(0, ty - 1).long()
        gx0 = gx0f.clamp(0, tx - 1).long()
        gy1 = (gy0f + 1).clamp(0, ty - 1).long()
        gx1 = (gx0f + 1).clamp(0, tx - 1).long()

        qf = q.reshape(-1)
        lut_flat = lut.reshape(ty * tx, 256)

        # Precompute the four (H,W) tile-index maps as 1-D flat indices.
        row0 = gy0.view(H, 1) * tx
        row1 = gy1.view(H, 1) * tx
        col0 = gx0.view(1, W)
        col1 = gx1.view(1, W)

        # Weighted accumulation, one corner at a time — never holds more
        # than one (1,1,H,W) gather in memory at once. Order matters for
        # numerical parity with the previous four-gather blend:
        #   out = (1-fy) * ((1-fx)*l00 + fx*l01) + fy * ((1-fx)*l10 + fx*l11)
        out = torch.empty(1, 1, H, W, device=dev, dtype=dt)

        flat = (row0 + col0).reshape(-1, 1)
        l = lut_flat[flat, qf.unsqueeze(1)].reshape(1, 1, H, W)
        w = (1.0 - fy) * (1.0 - fx)
        torch.mul(l, w, out=out)                        # out  = w00 * l00
        del l

        flat = (row0 + col1).reshape(-1, 1)
        l = lut_flat[flat, qf.unsqueeze(1)].reshape(1, 1, H, W)
        out.addcmul_(l, (1.0 - fy) * fx)                # out += w01 * l01
        del l

        flat = (row1 + col0).reshape(-1, 1)
        l = lut_flat[flat, qf.unsqueeze(1)].reshape(1, 1, H, W)
        out.addcmul_(l, fy * (1.0 - fx))                # out += w10 * l10
        del l

        flat = (row1 + col1).reshape(-1, 1)
        l = lut_flat[flat, qf.unsqueeze(1)].reshape(1, 1, H, W)
        out.addcmul_(l, fy * fx)                        # out += w11 * l11
        del l

        return out

    def _rgb_to_L_gpu(x):
        """RGB fp32 [0,1] -> perceptual lightness L in [0,1], on-GPU.
        Uses the Rec.709 luma weights then the same cube-root curve the LAB
        L* channel uses, so 'Improve Detail' adapts contrast perceptually
        like the old cv2.COLOR_BGR2LAB path did."""
        import torch
        r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
        y = 0.2126 * r + 0.7152 * g + 0.0722 * b
        eps = 216.0 / 24389.0
        kappa = 24389.0 / 27.0
        f = torch.where(y > eps, y.clamp_min(1e-12).pow(1.0 / 3.0),
                        (kappa * y + 16.0) / 116.0)
        return (116.0 * f - 16.0) / 100.0

    def _post_filters_gpu(t, params):
        """Post-inference cleanup, entirely on-GPU. Input/output are
        (1,3,H,W) fp32 RGB CUDA tensors in [0,1]. All four of these were
        CPU cv2/numpy ops on the FULL-RES upscaled frame before — the 2.7 s
        `post` bucket.

        dehalo  — edge-band median replacement. Canny thresholds are not
                  differentiable/portable, so the edge map is a Sobel
                  magnitude + threshold at the same relative cut; the band
                  (dilate-minus-edges) and the 3x3 box-blur 'median'
                  approximation blend with the same 0.35+0.55k strength.
        deblur  — Gaussian unsharp mask, same sigma and same weights.
        detail  — CLAHE on the L channel, same clip limit and 8x8 tiles.
        sharpen — Gaussian unsharp mask, identical math to before.

        BUGFIX (Error 1): all in-place math on `out` (add_/mul_/lerp_) is
        legal because _do_process wraps the whole per-frame path in a
        single torch.inference_mode() context. Nothing here mixes an
        inference-tensor input with an outside-inference-mode write."""
        import torch, torch.nn.functional as F
        out = t
        if params["dehalo"] > 0:
            k = params["dehalo"] / 100.0
            gray = (0.114 * out[:, 0:1] + 0.587 * out[:, 1:2]
                    + 0.299 * out[:, 2:3])              # B,G,R luma (cv2 order)
            kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]],
                              device=out.device, dtype=out.dtype).view(1, 1, 3, 3)
            ky = kx.transpose(-1, -2)
            gp = F.pad(gray, (1, 1, 1, 1), mode="reflect")
            gx = F.conv2d(gp, kx)
            gy = F.conv2d(gp, ky)
            del gp
            mag = (gx * gx + gy * gy).sqrt()
            del gx, gy
            # Old path: cv2.Canny(gray, 60, 180) on uint8 -> edges >= ~60/255.
            edges = (mag > (60.0 / 255.0 / 4.0)).to(out.dtype)  # sobel is ~4x gradient
            del mag
            band = -F.max_pool2d(-edges, 3, stride=1, padding=1)  # dilate 3x3
            band = (band - edges).clamp_min(0.0)
            del edges
            med = _box_blur_3x3(out)                    # 3x3-ish soft median
            mask = band * (0.35 + 0.55 * k)             # same strength
            del band
            out = out * (1.0 - mask) + med * mask
            del med, mask, gray
        if params["deblur"] > 0:
            k = params["deblur"] / 100.0
            sig = 0.4 + 0.9 * k
            blur = _gaussian_blur_gpu(out, sig)
            out = out * (1.0 + 0.35 * k) + blur * (-0.35 * k)   # same weights
            del blur
        if params["detail"] > 0:
            k = params["detail"] / 100.0
            L0 = _rgb_to_L_gpu(out)
            L = _clahe_l_gpu(L0, clip_limit=1.0 + 2.5 * k)      # same knob
            ratio = L / L0.clamp_min(1e-4)
            del L, L0
            out = (out * ratio).clamp(0.0, 1.0)         # rescale RGB by L ratio
            del ratio
        if params["sharpen"] > 0:
            k = params["sharpen"] / 100.0
            blur = _gaussian_blur_gpu(out, 1.2)
            amt = 0.2 + 1.2 * k
            out = out * (1.0 + amt) + blur * (-amt)     # identical to old math
            del blur
        return out

    def _infer_kernel_only(model, t, prof):
        """Model forward pass on an already-GPU tensor. Returns a GPU tensor.
        Measures ONLY the kernel via CUDA events; H2D and D2H are done once
        per frame elsewhere.

        Note: the enclosing scope in _do_process runs the whole per-frame
        path under torch.inference_mode(), which is strictly stronger than
        torch.no_grad() (no autograd view tracking, no version counters).
        We do NOT add a nested `with torch.no_grad():` here — it would be
        harmless but misleading."""
        import torch
        ev_start = torch.cuda.Event(enable_timing=True)
        ev_end = torch.cuda.Event(enable_timing=True)
        ev_start.record()
        y = model(t)
        ev_end.record()
        torch.cuda.synchronize()
        prof["infer_kernel"] += ev_start.elapsed_time(ev_end) / 1000.0  # ms -> s
        return y

    def _drain_stderr(pipe, buf):
        try:
            for line in iter(pipe.readline, b""):
                try:
                    buf.append(line.decode("utf-8", errors="replace").rstrip())
                except Exception:
                    buf.append(repr(line))
        except Exception:
            pass
        finally:
            try: pipe.close()
            except Exception: pass

    def _gpu_watcher(stop_evt, samples):
        """Background thread: snapshot nvidia-smi every 3s and stash a small
        rolling window. Cheap (~30ms per call), external process — will not
        distort the profile of the main loop."""
        while not stop_evt.is_set():
            s = _nvidia_smi_snapshot()
            if s:
                samples.append((time.time(), s))
            if stop_evt.wait(3.0):
                return

    def _do_process():
        import cv2, numpy as np, torch

        # ---- Allocator-fragmentation fix (Error 2, complementary to the
        # per-filter memory reductions below): let the caching allocator
        # grow its segments on demand instead of preallocating fixed-size
        # blocks that fragment across the large filter tensors. This has
        # to be set BEFORE the first CUDA allocation of the process to
        # take effect; if we've already allocated, the env var is a no-op
        # for this session (harmless) and will apply on the next kernel
        # start. Older PyTorch builds ignore the key and continue.
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

        in_path = MS["input_path"]
        out_dir = MS["workdir"] / "out"; out_dir.mkdir(exist_ok=True)
        stem = in_path.stem

        # 1. Probe input
        stage_lbl.value = "<b>Reading source metadata…</b>"
        fps = _ffprobe(in_path, ["-select_streams","v:0","-show_entries","stream=r_frame_rate","-of","csv=p=0"])
        num, den = (fps.split("/") + ["1"])[:2]
        fps_f = float(num) / float(den) if float(den) else 30.0
        nframes = int(_ffprobe(in_path, ["-select_streams","v:0","-count_packets","-show_entries","stream=nb_read_packets","-of","csv=p=0"]) or "0")
        logline("ok", f"Source: {fps_f:.3f} fps · ~{nframes or 'unknown'} frames.")

        # 2. Load model + GPU sanity check.
        stage_lbl.value = "<b>Loading model…</b>"
        logline("run", f"Loading {tier.value} model…")
        cuda_ok = torch.cuda.is_available()
        logline("info", f"torch.cuda.is_available() = {cuda_ok} · "
                        f"torch={torch.__version__} · "
                        f"device_count={torch.cuda.device_count() if cuda_ok else 0}")
        if not cuda_ok:
            raise RuntimeError("CUDA is not available — refusing to run on CPU. "
                               "Reconnect to a GPU runtime and re-run Step 1.")

        # cudnn.benchmark: pick fastest conv algo for the (fixed) input shape.
        torch.backends.cudnn.benchmark = True

        model, scale = _load_model(tier.value)
        gpu_name = torch.cuda.get_device_name(0)
        logline("ok", f"{tier.value} model loaded on {gpu_name} · native scale {scale}× · fp32.")

        # Ground truth for "is it actually on GPU?" — read the device off the
        # first parameter, not off what we THINK .cuda() did.
        try:
            first_param = next(model.model.parameters()) if hasattr(model, "model") else next(model.parameters())
        except Exception:
            first_param = None
        if first_param is None:
            raise RuntimeError("Could not read model parameters to confirm device placement.")
        logline("info", f"First model param device = {first_param.device} · dtype = {first_param.dtype}")
        if first_param.device.type != "cuda":
            raise RuntimeError(
                f"Model parameters ended up on {first_param.device} instead of "
                f"cuda after .cuda() — refusing to run (would be ~0.02 fps).")

        pre_snap = _nvidia_smi_snapshot()
        if pre_snap:
            logline("info", f"nvidia-smi (pre-run): {pre_snap}")

        params = dict(
            revert=s_revert.value, detail=s_detail.value, sharpen=s_sharpen.value,
            denoise=s_denoise.value, dehalo=s_dehalo.value, deblur=s_deblur.value,
            recover=s_recover.value,
        )
        logline("info",
                "Pipeline this run: frame is uploaded to GPU ONCE (infer_h2d), "
                "all pre/infer/recover/post stages run as GPU tensor ops inside "
                "torch.inference_mode(), and it is downloaded ONCE (infer_d2h) "
                "as packed BGR uint8 bytes for ffmpeg. No numpy/cv2 per frame. "
                "Bilateral/NLM/CLAHE rewritten as shift-and-accumulate to keep "
                "peak GPU memory bounded on 4K frames.")

        # 3. Open source + set up ffmpeg pipe.
        cap = cv2.VideoCapture(str(in_path))
        if not cap.isOpened():
            raise RuntimeError("OpenCV could not open the input video.")
        w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if w <= 0 or h <= 0:
            raise RuntimeError(f"OpenCV reported invalid source dimensions {w}x{h}.")

        # Output dims from the model's actual declared scale.
        out_w, out_h = w * scale, h * scale
        # x264 yuv420p needs even dims (see Bug 1 note in previous pass).
        crop_w = w - (out_w % 2 != 0)
        crop_h = h - (out_h % 2 != 0)
        if (crop_w, crop_h) != (w, h):
            logline("warn",
                    f"Source {w}x{h} at scale {scale}× would produce odd "
                    f"output ({out_w}x{out_h}); cropping source to "
                    f"{crop_w}x{crop_h} to keep yuv420p happy.")
            w, h = crop_w, crop_h
            out_w, out_h = w * scale, h * scale
        if out_w % 2 or out_h % 2:
            raise RuntimeError(f"Refusing to write odd output dims {out_w}x{out_h}.")

        stage_lbl.value = (f"<b>Upscaling {w}×{h} → {out_w}×{out_h} ({scale}×, fp32) …</b>")

        video_only = out_dir / f"{stem}_upscaled_noaudio.mp4"

        ffmpeg_cmd = [
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-f", "rawvideo",
            "-vcodec", "rawvideo",
            "-pix_fmt", "bgr24",
            "-s", f"{out_w}x{out_h}",
            "-r", f"{fps_f}",
            "-an",
            "-i", "-",
            "-c:v", "libx264",
            "-preset", "medium",
            "-crf", "16",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(video_only),
        ]
        logline("info", "ffmpeg: " + " ".join(ffmpeg_cmd))

        ff = subprocess.Popen(
            ffmpeg_cmd,
            stdin=subprocess.PIPE,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            bufsize=0,
        )
        stderr_buf = collections.deque(maxlen=200)
        stderr_thread = threading.Thread(
            target=_drain_stderr, args=(ff.stderr, stderr_buf), daemon=True,
        )
        stderr_thread.start()

        time.sleep(0.25)
        if ff.poll() is not None:
            stderr_thread.join(timeout=1.0)
            tail = "\n".join(stderr_buf) or "(no stderr captured)"
            raise RuntimeError(
                f"ffmpeg exited immediately (returncode={ff.returncode}). "
                f"stderr:\n{tail}"
            )

        expected_frame_bytes = out_w * out_h * 3

        frame_prog.max = max(nframes, 1)
        i = 0; t0 = time.time()

        # ---- Profiling state (same buckets as the profiling pass) ----
        stage_times = collections.defaultdict(float)
        report_at = {3, 5, 10, 15, 20, 30, 50, 100}
        buckets_ordered = ("read", "pre", "infer_h2d", "infer_kernel",
                           "infer_d2h", "recover", "post", "write")

        gpu_samples = collections.deque(maxlen=200)
        gpu_stop = threading.Event()
        gpu_thread = threading.Thread(
            target=_gpu_watcher, args=(gpu_stop, gpu_samples), daemon=True,
        )
        gpu_thread.start()

        last_ui_update = 0.0
        last_stdout_beat = 0.0
        last_gpu_log = 0.0

        # ---- GPU pipeline state ----
        need_pre  = (params["revert"] or params["denoise"])
        need_post = (params["dehalo"] or params["deblur"] or params["detail"] or params["sharpen"])
        need_rec  = params["recover"] > 0
        # Staging buffer for the H2D copy, lazily allocated on first frame.
        gpu_stage = None

        # ------------------------------------------------------------------
        # ERROR 1 FIX: wrap the ENTIRE per-frame path in a single
        # torch.inference_mode() so the model's inference-tensor outputs and
        # every subsequent in-place op (lerp_/add_/addcmul_) live in the
        # same context. This is the correct, permanent fix — not a per-call
        # .clone() sprinkled at the site of the first observed failure.
        # ------------------------------------------------------------------
        try:
            with torch.inference_mode():
                while True:
                    s = time.perf_counter()
                    ok, frame = cap.read()
                    if not ok:
                        break
                    if frame.shape[1] != w or frame.shape[0] != h:
                        frame = frame[:h, :w]
                    stage_times["read"] += time.perf_counter() - s

                    # ---- H2D: ONE upload per frame (uint8 BGR HWC) ----
                    s = time.perf_counter()
                    if (gpu_stage is None
                            or gpu_stage.shape[0] != frame.shape[0]
                            or gpu_stage.shape[1] != frame.shape[1]):
                        gpu_stage = torch.empty(frame.shape,
                                                dtype=torch.uint8,
                                                device="cuda")
                    gpu_stage.copy_(torch.from_numpy(frame), non_blocking=False)
                    t_in = _bgr_u8_to_rgb_f32(gpu_stage)
                    torch.cuda.synchronize()
                    stage_times["infer_h2d"] += time.perf_counter() - s

                    # ---- pre (GPU) ----
                    s = time.perf_counter()
                    if need_pre:
                        t_in_new = _pre_filters_gpu(t_in, params)
                        # _pre_filters_gpu returns a fresh tensor when either
                        # branch runs; free the pre-filter input to keep peak
                        # memory flat on HIGH tier.
                        if t_in_new is not t_in:
                            del t_in
                        t_in = t_in_new
                        torch.cuda.synchronize()
                    stage_times["pre"] += time.perf_counter() - s

                    # ---- infer_kernel (GPU, CUDA-event timed) ----
                    y = _infer_kernel_only(model, t_in, stage_times)

                    # ---- recover (GPU) ----
                    s = time.perf_counter()
                    if need_rec:
                        y = _recover_blend_gpu(y, t_in, params["recover"], scale)
                        torch.cuda.synchronize()
                    # If we don't need the low-res input any more, release
                    # it now — before the ~4x-larger post-tensors allocate.
                    if not need_rec:
                        del t_in
                    stage_times["recover"] += time.perf_counter() - s

                    # ---- post (GPU) ----
                    s = time.perf_counter()
                    if need_post:
                        y = _post_filters_gpu(y, params)
                        torch.cuda.synchronize()
                    stage_times["post"] += time.perf_counter() - s

                    # ---- D2H: ONE download per frame (uint8 BGR bytes) ----
                    s = time.perf_counter()
                    out_u8 = _rgb_f32_to_bgr_u8(y)          # GPU tensor, packed
                    # Release the fp32 upscale tensor immediately; the D2H
                    # copy below only needs `out_u8`. Also release the
                    # low-res source if recover kept it alive.
                    del y
                    if need_rec:
                        try:
                            del t_in
                        except NameError:
                            pass
                    buf = out_u8.cpu().numpy().tobytes()    # single D2H + bytes
                    del out_u8
                    stage_times["infer_d2h"] += time.perf_counter() - s

                    if len(buf) != expected_frame_bytes:
                        raise RuntimeError(
                            f"Frame {i}: byte length {len(buf)} != expected "
                            f"{expected_frame_bytes} (source {h}x{w}, model "
                            f"native scale {scale}×).")

                    s = time.perf_counter()
                    try:
                        ff.stdin.write(buf)
                    except BrokenPipeError:
                        ff.wait(timeout=2.0)
                        stderr_thread.join(timeout=1.0)
                        tail = "\n".join(stderr_buf) or "(no stderr captured)"
                        raise RuntimeError(
                            f"Broken pipe while writing frame {i} to ffmpeg "
                            f"(returncode={ff.returncode}). stderr:\n{tail}"
                        )
                    stage_times["write"] += time.perf_counter() - s
                    i += 1

                    # After the first frame, once cudnn.benchmark has
                    # picked its algo and the shape caches are warm,
                    # release the transient scratch ONCE. Not per-frame.
                    if i == 1:
                        torch.cuda.empty_cache()

                    # ---- Per-frame profile reports (same format as before) ----
                    if i in report_at:
                        total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
                        parts = []
                        for k in buckets_ordered:
                            v = stage_times[k]
                            parts.append(f"{k}={v/i*1000:.1f}ms ({v/total*100:.0f}%)")
                        fps_now = i / max(time.time() - t0, 1e-6)
                        logline("perf",
                                f"[frame {i}] {fps_now:.3f} fps · " + " · ".join(parts))

                    # ---- Periodic GPU util log ----
                    now = time.time()
                    if (now - last_gpu_log) >= 6.0 and gpu_samples:
                        _, snap = gpu_samples[-1]
                        logline("perf", f"nvidia-smi: {snap} @ frame {i}")
                        last_gpu_log = now

                    # ---- UI heartbeat ----
                    if i == 1 or (now - last_ui_update) >= 0.5 or i == nframes:
                        frame_prog.value = min(i, frame_prog.max)
                        elapsed = now - t0
                        fps_now = i / max(elapsed, 1e-6)
                        eta = (nframes - i) / fps_now if (nframes and fps_now > 0) else 0
                        eta_str = f" · ETA {int(eta//60)}m{int(eta%60):02d}s" if eta else ""
                        frame_pct.value = f"{i}/{nframes or '?'} · {fps_now:.2f} fps{eta_str}"
                        last_ui_update = now
                    if (now - last_stdout_beat) >= 10.0:
                        print(f"[motionsalt] frame {i}/{nframes or '?'} "
                              f"({i/max(now-t0,1e-6):.2f} fps)", flush=True)
                        last_stdout_beat = now
        finally:
            cap.release()
            try:
                if ff.stdin and not ff.stdin.closed:
                    ff.stdin.close()
            except Exception:
                pass
            gpu_stop.set()

        rc = ff.wait()
        stderr_thread.join(timeout=2.0)
        gpu_thread.join(timeout=4.0)
        if rc != 0:
            tail = "\n".join(stderr_buf) or "(no stderr captured)"
            raise RuntimeError(
                f"ffmpeg exited with code {rc} after writing {i} frames. "
                f"stderr:\n{tail}"
            )

        # ---- Final summary: authoritative per-stage table ----
        total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
        wallclock = time.time() - t0
        final_fps = i / max(wallclock, 1e-6)
        logline("ok", f"Upscaled {i} frames in {wallclock:.1f}s ({final_fps:.3f} fps).")

        summary_lines = [
            f"<b>Per-stage average (ms/frame) over {i} frames, {final_fps:.3f} fps wall:</b>"
        ]
        for k in buckets_ordered:
            v = stage_times[k]
            summary_lines.append(
                f"&nbsp;&nbsp;{k:14s}: {v/i*1000:8.1f} ms/frame  ({v/total*100:4.1f}%)"
            )
        if gpu_samples:
            utils = []
            for _, s in gpu_samples:
                try:
                    utils.append(int(s.split("gpu=")[1].split("%")[0]))
                except Exception:
                    pass
            if utils:
                summary_lines.append(
                    f"&nbsp;&nbsp;GPU util range: min={min(utils)}% "
                    f"max={max(utils)}% avg={sum(utils)//len(utils)}% "
                    f"across {len(utils)} samples"
                )
        logline("perf", "<br>".join(summary_lines).replace("\n", "<br>"))

        # 4. Mux original audio back in.
        stage_lbl.value = "<b>Muxing original audio…</b>"
        with_audio = out_dir / f"{stem}_upscaled.mp4"
        rc = subprocess.run(
            ["ffmpeg","-y","-hide_banner","-loglevel","error",
             "-i",str(video_only),"-i",str(in_path),
             "-map","0:v:0","-map","1:a:0?","-c:v","copy","-c:a","aac","-b:a","192k",
             "-shortest", str(with_audio)],
            capture_output=True, text=True,
        )
        if rc.returncode != 0:
            if rc.stderr:
                logline("warn", f"Audio mux failed: {rc.stderr.strip().splitlines()[-1] if rc.stderr.strip() else 'unknown'}")
            shutil.move(str(video_only), str(with_audio))
            logline("warn","Source had no audio track (or codec mismatch); output is silent.")
        else:
            try: os.remove(video_only)
            except OSError: pass
            logline("ok","Audio muxed back in.")

        final = with_audio

        # 5. Optional 1080p-height downscale.
        if cb_1080.value:
            stage_lbl.value = "<b>Downscaling to 1080p height (aspect-preserving)…</b>"
            down = out_dir / f"{stem}_upscaled_1080p.mp4"
            rc = subprocess.run(
                ["ffmpeg","-y","-hide_banner","-loglevel","error",
                 "-i",str(final),
                 "-vf","scale=-2:1080:flags=lanczos",
                 "-c:v","libx264","-preset","medium","-crf","17","-pix_fmt","yuv420p",
                 "-c:a","copy",
                 str(down)],
                capture_output=True, text=True,
            )
            if rc.returncode == 0:
                final = down
                logline("ok","Downscaled to 1080p height, aspect ratio preserved.")
            else:
                last = rc.stderr.strip().splitlines()[-1] if rc.stderr.strip() else "unknown"
                logline("warn", f"1080p downscale failed ({last}) — keeping full-res output.")

        MS["output_path"] = final
        stage_lbl.value = f"<b>Done.</b> Output: <code>{final.name}</code>"
        logline("ok", f"Ready for Step 4. File: <b>{final.name}</b> · {final.stat().st_size/1e6:.1f} MB.")

    start.on_click(start_processing)

    display(widgets.VBox([
        hdr, tier,
        s_revert, s_detail, s_sharpen, s_denoise, s_dehalo, s_deblur, s_recover,
        cb_1080,
        widgets.HTML("<hr style='border:none;border-top:1px solid #e2e8f0;margin:8px 0;'>"),
        start,
        stage_lbl,
        widgets.HBox([frame_prog, frame_pct]),
        log,
    ], layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


## Step 4 — Download

Click the button. Your browser downloads the result directly. No public/shareable link is generated — the file only exists on your machine and the Colab VM.

In [ ]:
#@title ⬇️ Step 4 — Download result { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("output_path"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 3 first (Configure & process).</div>'))
else:
    out = MS["output_path"]
    hdr = widgets.HTML(
        f'<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Your file is ready</div>'
        f'<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;"><code>{out.name}</code> · {out.stat().st_size/1e6:.1f} MB</div>'
    )
    btn = widgets.Button(description="⬇️ Download Result", button_style="primary",
                         layout=widgets.Layout(width="240px", height="46px"))
    status = widgets.HTML("")

    def on_click(_):
        from google.colab import files
        btn.disabled = True
        status.value = '<div style="font-family:sans-serif;color:#475569;font-size:12.5px;">Preparing browser download…</div>'
        files.download(str(out))
        status.value = '<div style="padding:10px;background:#dcfce7;color:#166534;border-radius:8px;font-family:sans-serif;">✅ Download started in your browser.</div>'
        btn.disabled = False

    btn.on_click(on_click)
    display(widgets.VBox([hdr, btn, status],
                         layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


---

<div align="center" style="font-family:-apple-system,Segoe UI,sans-serif;color:#64748b;font-size:12px;padding:10px;">
MOTIONSALT Upscaler · MIT-licensed wrapper · powered by
<a href="https://github.com/the-database/mpv-upscale-2x_animejanai">AnimeJaNai V3</a> and
<a href="https://github.com/xinntao/Real-ESRGAN">Real-ESRGAN AnimeVideo v3</a>.<br>
Source: <a href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
</div>